## 0.  MOTIVATION

We independently replicate the findings of the paper "A data-driven approach to predict the success of bank telemarketing" by S.Moro, P.Cortez,P.Rita (2014), DOI:10.1016/j.dss.2014.03.001. The authors predict the success of telemarketing calls for selling bank long-term
deposits for a Portuguese bank, with data collected from 2008 to 2013. The initial dataset contained 150 features related with bank client, product and social-economic attributes, but the authors managed to isolate 21 features (+ the target, i.e. if a client subscribed a long-term deposit) which are sufficient for accurately predicting the success rate. This reduced dataset is the one used in this notebook. More precisely, we take the dataset from https://archive.ics.uci.edu/dataset/222/bank+marketing, specifically dataset 1), i.e. bank-additional-full.csv. This is not *exactly* the dataset used in the paper, but it is very close.

The dataset is indeed enriched by the addition of five new social and economic features/attributes which we specify, and whose inclusion greatly improves the prediction of whether a client subscribes a long-term loan.

We compare the results obtained with and without the additional social features, als comparing it to the results presented in the article.

In [18]:
## 1. Import relevant packages

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (roc_curve, precision_recall_curve, roc_auc_score,
                             average_precision_score, confusion_matrix, f1_score)

In [21]:
## 2. Load the dataset and define training and test sets

df = pd.read_csv("bank-additional-full.csv", delimiter=";")
FEATURES = df.columns.values.tolist()[0:20]
FEATURES.remove("duration")         #As noted in the bank-additional-names.txt file accompanying the dataset the duration is not known a priori and it                                                    should not be used in training predictive models.
FEATURES_reduced = FEATURES[0:14].copy()
TARGET = "deposit"
df.rename(columns={'y': "deposit"}, inplace=True)

X, X_reduced, y = df[FEATURES], df[FEATURES_reduced],df[TARGET]

['age',
 'job',
 'marital',
 'education',
 'default',
 'housing',
 'loan',
 'contact',
 'month',
 'day_of_week',
 'campaign',
 'pdays',
 'previous',
 'poutcome']

The dataset is based on "Bank Marketing" UCI dataset (please check the description at: http://archive.ics.uci.edu/ml/datasets/Bank+Marketing). The data contains five social and economic features/attributes that, according to the authors, increase the model precision.

The raw dataset has 41188 rows and 21 features, with the last one (renamed "deposit") being the target feature, which determines if a person subscribes a bank deposit or not. We drop the feature "duration" which measures the last contact duration, in seconds, of  call with the potential client.

We now describe each feature to better understand the setting. Features refer to both

- "age" : (Numerical).
- "job" : (Categorical: "admin.","blue-collar","entrepreneur","housemaid","management","retired","self-employed","services","student","technician","unemployed","unknown").
- "marital" : (Categorical: "divorced","married","single","unknown").
- "education" : (Categorical: "basic.4y","basic.6y","basic.9y","high.school","illiterate","professional.course","university.degree","unknown").
- "default" : Has default credit? (Binary: "yes", "no", "unknown").
- "housing" : Has housing loan? (Categorical: "no","yes","unknown").
- "loan" : Has personal loan? (Categorical: "no","yes","unknown").
- "contact" : Contact communication type (Categorical: "cellular","telephone").
- "month": (Categorical: "jan", "feb", "mar", ..., "nov", "dec").
- "day_of_week" : (Categorical: "mon","tue","wed","thu","fri").
- "campaign" : Number of contacts performed during this campaign and for this client (Numeric).
- "pdays": number of days that passed by after the client was last contacted from a previous campaign (Numeric; 999 means client was not previously contacted).
- "previous": Number of contacts performed before this campaign and for this client (Numeric).
- "poutcome": Outcome of the previous marketing campaign (Categorical: "failure","nonexistent","success").





